<a href="https://colab.research.google.com/github/ANUDHEERPARIMI/PyTorch/blob/PyTorch/fasion_minist_dataset_prac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,Dataset
import torch.nn as nn
import torch

In [2]:
torch.manual_seed(42)

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/fashionmnist


In [4]:
import os
path = "/kaggle/input/fashionmnist"
print(os.listdir(path))


['t10k-labels-idx1-ubyte', 't10k-images-idx3-ubyte', 'fashion-mnist_test.csv', 'fashion-mnist_train.csv', 'train-labels-idx1-ubyte', 'train-images-idx3-ubyte']


In [5]:
df=pd.read_csv("/kaggle/input/fashionmnist/fashion-mnist_train.csv")

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [7]:
df.shape

(60000, 785)

In [8]:
X=df.iloc[:,1:].values
y=df.iloc[:,0].values

In [9]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [10]:
y

array([2, 9, 6, ..., 8, 8, 7])

In [11]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [12]:
X_train=X_train/255.0
X_test=X_test/255.0

In [13]:
class CustomDataset(Dataset):
  def __init__(self,features,label):
    self.features=torch.tensor(features,dtype=torch.float32)
    self.label=torch.tensor(label,dtype=torch.long)
  def __len__(self):
    return self.features.shape[0]
  def __getitem__(self,idx):
    return self.features[idx],self.label[idx]

In [14]:
train_dataset=CustomDataset(X_train,y_train)
test_dataset=CustomDataset(X_test,y_test)

In [15]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)


In [16]:
class MyNN(nn.Module):
  def __init__(self,features):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Linear(features, 128),
      nn.ReLU(),
      nn.Linear(128, 64),
      nn.ReLU(),
      nn.Linear(64, 10)
  )

  def forward(self, x):
      return self.layers(x)



We Are doing regularization as the cuurrect network is giving accuracy

Dropout

In [17]:
class MyNN(nn.Module):
  def __init__(self,features):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Linear(features, 128),
      nn.ReLU(),
      nn.Dropout(0.3),
      nn.Linear(128, 64),
      nn.ReLU(),
      nn.Dropout(0.3),
      nn.Linear(64, 10)
  )

  def forward(self, x):
      return self.layers(x)


BatchNorm

In [18]:
class MyNN(nn.Module):
  def __init__(self,features):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Linear(features, 128),
      nn.BatchNorm1d(128),
      nn.ReLU(),
      nn.Dropout(0.3),
      nn.Linear(128, 64),
      nn.BatchNorm1d(64),
      nn.ReLU(),
      nn.Dropout(0.3),
      nn.Linear(64, 10)
  )

  def forward(self, x):
      return self.layers(x)


In [19]:
epochs=100
learning_rate=0.1

In [20]:
model=MyNN(X_train.shape[1])
model=model.to(device)

criterian = nn.CrossEntropyLoss()

optmizer = torch.optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)

In [21]:
for epoch in range(epochs):
  total_loss=0
  for batch_features,batch_labels in train_loader:

    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

    y_pred=model(batch_features)

    loss = criterian(y_pred,batch_labels)

    optmizer.zero_grad()

    loss.backward()

    optmizer.step()

    total_loss+=loss.item()
  print(f"{epoch+1} for this this {total_loss/len(train_loader)}")

1 for this this 0.6249084657629331
2 for this this 0.49199690653880435
3 for this this 0.45562089485426743
4 for this this 0.43380642544229825
5 for this this 0.41715061584611735
6 for this this 0.40564093277355034
7 for this this 0.3941608931571245
8 for this this 0.38580174928406874
9 for this this 0.3743983890265226
10 for this this 0.3725726637095213
11 for this this 0.36783315147956214
12 for this this 0.3572052289446195
13 for this this 0.35052060889204345
14 for this this 0.3449219484726588
15 for this this 0.34472562207778296
16 for this this 0.33732124184072015
17 for this this 0.3344038988550504
18 for this this 0.3302020480086406
19 for this this 0.33063985937833784
20 for this this 0.3262277270356814
21 for this this 0.3208496819138527
22 for this this 0.3183093272894621
23 for this this 0.3225850373158852
24 for this this 0.31459670132398604
25 for this this 0.31343053522954384
26 for this this 0.31424527982374034
27 for this this 0.3107087447295586
28 for this this 0.3090

In [22]:
model.eval()

MyNN(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [23]:
total=0
correct=0

with torch.no_grad():
  for batch_features,batch_labels in test_loader:

    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

    outputs = model(batch_features)

    _,predicted=torch.max(outputs,1)


    total = total + batch_labels.shape[0]

    correct = correct + (predicted==batch_labels).sum().item()


In [24]:
print(correct/total)

0.88325


In [25]:
total

12000

In [26]:
y_test.shape

(12000,)

In [27]:
for x,y in test_loader:
  print(y)
  break

tensor([7, 8, 8, 5, 9, 1, 2, 6, 6, 2, 5, 0, 7, 1, 6, 0, 6, 2, 9, 1, 2, 4, 8, 0,
        4, 9, 1, 0, 0, 5, 1, 6])
